<a href="https://colab.research.google.com/github/Maximi652/efficient-slm-architectures/blob/main/LotteryTicketSLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch, os, gc
import torch.nn.utils.prune as prune
from transformers import AutoTokenizer, AutoModelForCausalLM

# Speicherpfad und Device
model_name = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B"
device = "cuda" if torch.cuda.is_available() else "cpu"
init_path = "./init_weights"

# Lade Modell & Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda" if torch.cuda.is_available() else "cpu",
    trust_remote_code=True
)

model.eval()
model = torch.compile(model)
model.to(device)

# Hilfsfunktion: prunebare Keys (Linear-Layer) extrahieren
def get_prunable_keys(model):
    return [
        k for k in model.state_dict().keys()
        if 'weight' in k and any(x in k for x in ['mlp', 'self_attn', 'attn', 'linear', 'dense'])
    ]

# Initialgewichte speichern (nur prunebare Layer, FP16, auf Disk)
def save_init_weights(model, keys, path=init_path):
    os.makedirs(path, exist_ok=True)
    for k in keys:
        tensor = model.state_dict()[k].half().cpu()
        torch.save(tensor, os.path.join(path, f"{k.replace('.', '_')}.pt"))

def load_init_weight(key, path=init_path):
    return torch.load(os.path.join(path, f"{key.replace('.', '_')}.pt")).float()

def prune_linear_layers_cpu(model, amount):
    device = next(model.parameters()).device
    model.cpu()
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            prune.l1_unstructured(module, name='weight', amount=amount)
    model.to(device)
    torch.cuda.empty_cache()  # Speicher aufräumen
    gc.collect()  # Garbage Collector aufrufen


def reset_weights_to_init(model, keys, path='./init_weights'):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear) and hasattr(module, 'weight_orig') and hasattr(module, 'weight_mask'):
            param_key = f"{name}.weight_orig"
            if param_key in keys:
                orig = torch.load(os.path.join(path, f"{param_key.replace('.', '_')}.pt")).to(module.weight_orig.device)
                mask = module.weight_mask
                with torch.no_grad():
                    module.weight_orig.data = module.weight_orig.data * (1-mask) + orig * mask

def remove_pruning(model):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            # Nur wenn Mask/Orig vorhanden
            if hasattr(module, "weight_mask") and hasattr(module, "weight_orig"):
                prune.remove(module, "weight")

prunable_keys = get_prunable_keys(model)
save_init_weights(model, prunable_keys, path=init_path)

n_iter = 5
prune_each = 1 - (1-0.5)**(1/5)    # ≈ 0.13, damit 5 Iterationen zusammen 50%

for i in range(n_iter):
    print(f"=== LTH Iteration {i+1}/{n_iter} | Prune {prune_each*100:.2f}% ===")
    prune_linear_layers_cpu(model, amount=prune_each)
    reset_weights_to_init(model, prunable_keys, path=init_path)
    remove_pruning(model)

# Originale Maske entfernen
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        if hasattr(module, "weight_mask") and hasattr(module, "weight_orig"):
            prune.remove(module, "weight")

output_dir = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B_LTH"

# PyTorch-Checkpoint
model.save_pretrained(output_dir, safe_serialization=True)

# Tokenizer speichern
tokenizer.save_pretrained(output_dir)

In [ ]:
# ==== 10. Metriken zeigen ====
def compute_pruned_stats(model):
    total_params = 0
    nonzero_params = 0
    for name, param in model.named_parameters():
        if param.requires_grad and param.dim() > 1 and "weight" in name:
            total_params += param.numel()
            nonzero_params += torch.count_nonzero(param).item()
    zero_params = total_params - nonzero_params
    sparsity = 100.0 * zero_params / total_params
    compression_ratio = total_params / nonzero_params if nonzero_params > 0 else float("inf")
    print(f"Gesamtparameter: {total_params:,}")
    print(f"Aktive Parameter (<> 0): {nonzero_params:,}")
    print(f"Sparsity: {sparsity:.2f}%")
    print(f"Komprimierungsrate: {compression_ratio:.2f}x")
    return total_params, nonzero_params, sparsity, compression_ratio

compute_pruned_stats(model)

# Inferenz

In [ ]:
# Install und Imports
!pip install -q transformers datasets

import json
from datasets import Dataset
import torch
from torch import inference_mode
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

# Pfade und Gerät
model_name = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B_LTH"
test_json_path = "/content/drive/MyDrive/Colab Notebooks/12b_golden_testdata.json"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Basis-Modell und LoRA-Adapter
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cuda",
    trust_remote_code=True
).to(device)

gen_conf = GenerationConfig(
    max_new_tokens=200,   # Limitiere die generierten Tokens
    do_sample=False,      # Greedy- statt Sample-Decoding
    use_cache=True       # Aktiviert KV-Caching
)

# Testdaten parsen und formatieren
def format_entry_chat(entry):

    qtext = entry["body"].strip()
    qtype = entry["type"].lower()
    # ctx = "\n".join(s["text"].strip() for s in list(entry["snippets"]))
    results = []

    # Ausformulierte Antwort
    ideal_ans = entry["ideal_answer"][0].strip()
    if ideal_ans:
        if qtype == "yesno":
            # user_msg = f"Question: {qtext}\nContext:\n{ctx}\nProvide one-sentence ideal answer in English starting with 'Yes,' or 'No,'."
            user_msg = f"Question: {qtext}\nProvide one-sentence ideal answer in English starting with 'Yes,' or 'No,'."
        else:
            # user_msg = f"Question: {qtext}\nContext:\n{ctx}\nProvide an ideal answer in English (one paragraph, max 200 words, full sentences)."
            user_msg = f"Question: {qtext}\nProvide an ideal answer in English (one paragraph, max 200 words, full sentences)."
        # messages = [
        #     {"role": "system", "content": "/no_think"},
        #     {"role": "user", "content": user_msg},
        # ]
        # text = tokenizer.apply_chat_template(
        #     messages,
        #     tokenize=False,
        #     add_generation_prompt=True,
        #     enable_thinking=False
        # )#
        text = (
        "<|im_start|>system\n"
        "/no_think\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user_msg}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        )
        results.append({"text": text})
        print("Prompt:", repr(text))

    # Kurzantwort
    try:

        answer = entry["exact_answer"]

    except KeyError:

        answer = ""

    exact = answer
    flat_exact = [item[0] if isinstance(item, list) and item else item for item in exact]
    exact_ans = ", ".join(flat_exact).strip()
    if exact_ans:
        if qtype == "yesno":
            # user_msg = f"Question: {qtext}\nContext:\n{ctx}\nAnswer only 'yes' or 'no', in English, no extras."
            user_msg = f"Question: {qtext}\nAnswer only 'yes' or 'no', in English, no extras."
        elif qtype == "factoid":
            # user_msg = f"Question: {qtext}\nContext:\n{ctx}\nProvide up to 5 keywords, comma-separated, in English, no commentary."
            user_msg = f"Question: {qtext}\nProvide up to 5 keywords, comma-separated, in English, no commentary."
        elif qtype == "list":
            # user_msg = f"Question: {qtext}\nContext:\n{ctx}\nProvide a comma-separated list of relevant items, in English, no filler words."
            user_msg = f"Question: {qtext}\nProvide a comma-separated list of relevant items, in English, no filler words."
        else:
            # user_msg = f"Question: {qtext}\nContext:\n{ctx}\nProvide a brief answer in English."
            user_msg = f"Question: {qtext}\nProvide a brief answer in English."
        messages = [
            {"role": "system", "content": "/no_think"},
            {"role": "user", "content": user_msg},
        ]
        # text = tokenizer.apply_chat_template(
        #     messages,
        #     tokenize=False,
        #     add_generation_prompt=True,
        #     enable_thinking=False
        # )
        # print("Prompt:", repr(text))
        text = (
        "<|im_start|>system\n"
        "/no_think\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user_msg}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        )
        results.append({"text": text})
        print("Prompt:", repr(text))

    return results

with open(test_json_path, "r", encoding="utf-8") as f:
    raw_test = json.load(f)["questions"]

# Flatten
formatted_test = []
for entry in raw_test:
    formatted_test.extend(format_entry_chat(entry))

def extract_qwen_answer(output_text):
    lines = output_text.splitlines()
    answer_lines = []
    in_answer = False
    skip_think = False
    for idx, line in enumerate(lines):
        # Start nach 'assistant'
        if not in_answer and line.strip() == "assistant":
            in_answer = True
            # Optional: falls  folgt, überspringen
            if idx + 1 < len(lines) and lines[idx + 1].strip() == "":
                skip_think = True
            continue
        if in_answer:
            if skip_think:
                if line.strip() == "":
                    skip_think = False
                continue
            answer_lines.append(line)
    # Antwortzeilen zusammenbauen
    return "\n".join(answer_lines).strip()

# Inference
def generate_answer(prompt_text, ):

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        padding=False        # gar nicht auf max length padden
    ).to(device)

    # Sampling-Parameter anpassen
    with inference_mode():
        output_ids = model.generate(
            **inputs,
            generation_config=gen_conf,
            pad_token_id=tokenizer.eos_token_id
        )

    # 3) Volltext decodieren & Prompt entfernen
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Output vergleichen
    print("="*80)
    print("PROMPT:")
    print(repr(prompt_text))
    print("OUTPUT:")
    print(repr(output_text))
    print("="*80)

    final_answer = extract_qwen_answer(output_text)

    return final_answer

# Loop über den Testdatensatz
results = []

for idx, ex in enumerate(formatted_test):
    prompt = ex["text"]
    pred = generate_answer(prompt)
    results.append({
        "index": idx,
        # "question": ex["question"],
        "prompt": prompt,
        "prediction": pred
    })

    print(pred)
    print("-" * 50)
    if idx % 20 == 0:
        print(f"Processed {idx}/{len(formatted_test)}")

# 8. Ergebnisse speichern oder auswerten
import json
with open("/content/drive/MyDrive/Colab Notebooks/test_predictions_LTH.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Inference abgeschlossen – Ergebnisse in test_predictions_LTH.json gespeichert.")